# Fraud Rules Evaluation - ClickHouse Direct Queries

This notebook executes fraud detection rules directly in ClickHouse and evaluates their performance.
Each rule is executed separately and results are stored in a pandas DataFrame for analysis.

## 1. Setup and Imports

In [1]:
import pandas as pd
from clickhouse_driver import Client
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries imported successfully")

✅ Libraries imported successfully


## 2. ClickHouse Connection Configuration

In [2]:
# ClickHouse configuration
CLICKHOUSE_CONFIG = {
    'host': 'localhost',
    'port': 9000,
    'database': 'public',
    'user': 'default',
    'password': 'DfsTeChB1'
}

# Create ClickHouse client
client = Client(**CLICKHOUSE_CONFIG)

# Test connection
result = client.execute('SELECT version()')
print(f"✅ Connected to ClickHouse version: {result[0][0]}")

✅ Connected to ClickHouse version: 25.10.1.3796


## 3. Define Date Range and Parameters

In [3]:
# Date range for analysis
START_DATE = '2025-07-01'
END_DATE = '2025-07-30'

print(f"📅 Analysis Period: {START_DATE} to {END_DATE}")
print(f"📊 Table: public.stixor_fraud_features_distributed")
print(f"🔍 Filter: Customer Account only")

📅 Analysis Period: 2025-07-01 to 2025-07-30
📊 Table: public.stixor_fraud_features_distributed
🔍 Filter: Customer Account only


## 4. Define All Rule Queries

## 4a. Rule Categories Overview

The following 24 fraud detection rules are organized into these categories:

### 🔢 Base Rules (Rules 1-6)
- **Rule 1**: New Account (<30 days) with High Amount (>$10,000)
- **Rule 2**: Very New Account (<1 hour) with High Amount (>$10,000)
- **Rule 3**: High Transaction Velocity (>10 txns in 1 hour)
- **Rule 4**: Unusual Amount Pattern (3x max amount in last 30 days)
- **Rule 5**: Multiple Channels in Short Time (>1 channel in 30 min)
- **Rule 6**: Off-Peak Hours Transaction (1 AM - 4 AM)

### ⏰ Time-Based Rules (2 rules)
- **Weekend Night**: Transactions on weekends between 10 PM - 4 AM
- **Unusual Hour**: Transactions at 2 AM, 3 AM, or 4 AM

### 🆕 Account Age Rules (4 rules)
- **New High Amount**: Account <30 days with amount >$500
- **Very New Any**: Account <7 days, any transaction
- **Immediate Activity**: First transaction <1 hour after registration
- **Dormant High**: Dormant >30 days, then high amount transaction

### 💰 Amount Rules (6 rules)
- **High Value**: Transaction amount >$5,000
- **Round Number**: Amounts in multiples of 1000
- **Structuring**: Amounts just below thresholds (4900-4999, 9900-9999)
- **Amount Spike**: Current amount >3x historical average
- **3 StdDev**: Amount >3 standard deviations from average

### 📱 Channel Rules (2 rules)
- **PGW New High**: Payment Gateway + new account + high amount
- **Mobile New High**: Mobile App + new account + high amount

### 🔄 Behavioral Rules (2 rules)
- **New Recipient High**: First transaction to recipient with high amount
- **Amount Jump**: Sudden 5x increase in amount (when avg <$1,000)

### 🎯 Recipient Rules (2 rules)
- **Money Mule**: Recipient receives from >10 senders in 24 hours
- **High Amount 1hr**: Recipient receives >$10,000 in 1 hour

**Total: 24 comprehensive fraud detection rules**


In [4]:
# Dictionary to store all rule queries
rule_queries = {
    'rule_1': """
    WITH rule_1_calc AS (
        SELECT 
            trans_id,
            fraud_flag,
            CASE 
                WHEN dateDiff('day', toDate(mbar_registered_date_time), cutoff_date) < 30 
                    AND trx_amt > 10000 
                THEN 1 ELSE 0 
            END AS rule_1_flag
        FROM public.stixor_fraud_features_distributed
        WHERE cutoff_date BETWEEN '{start_date}' AND '{end_date}'
            AND mbar_account_type_name = 'Customer Account'
    )
    SELECT 
        'Rule 1: New Account High Amount' AS rule_name,
        countIf(rule_1_flag = 1 AND fraud_flag = 1) AS true_positives,
        countIf(rule_1_flag = 1 AND fraud_flag = 0) AS false_positives,
        countIf(rule_1_flag = 0 AND fraud_flag = 1) AS false_negatives,
        countIf(rule_1_flag = 0 AND fraud_flag = 0) AS true_negatives,
        round(countIf(rule_1_flag = 1 AND fraud_flag = 1) * 100.0 / nullIf(countIf(rule_1_flag = 1), 0), 4) AS precision,
        round(countIf(rule_1_flag = 1 AND fraud_flag = 1) * 100.0 / nullIf(countIf(fraud_flag = 1), 0), 4) AS recall,
        round(2 * (countIf(rule_1_flag = 1 AND fraud_flag = 1) * 100.0 / nullIf(countIf(rule_1_flag = 1), 0)) * 
                  (countIf(rule_1_flag = 1 AND fraud_flag = 1) * 100.0 / nullIf(countIf(fraud_flag = 1), 0)) /
              nullIf((countIf(rule_1_flag = 1 AND fraud_flag = 1) * 100.0 / nullIf(countIf(rule_1_flag = 1), 0)) + 
                     (countIf(rule_1_flag = 1 AND fraud_flag = 1) * 100.0 / nullIf(countIf(fraud_flag = 1), 0)), 0), 4) AS f1_score,
        countIf(rule_1_flag = 1) AS total_flagged
    FROM rule_1_calc
    """,
    
    'rule_2': """
    WITH rule_2_calc AS (
        SELECT 
            trans_id,
            fraud_flag,
            CASE 
                WHEN dateDiff('hour', mbar_registered_date_time, trans_initiate_time) < 1 
                    AND trx_amt > 10000 
                THEN 1 ELSE 0 
            END AS rule_2_flag
        FROM public.stixor_fraud_features_distributed
        WHERE cutoff_date BETWEEN '{start_date}' AND '{end_date}'
            AND mbar_account_type_name = 'Customer Account'
    )
    SELECT 
        'Rule 2: Very New Account' AS rule_name,
        countIf(rule_2_flag = 1 AND fraud_flag = 1) AS true_positives,
        countIf(rule_2_flag = 1 AND fraud_flag = 0) AS false_positives,
        countIf(rule_2_flag = 0 AND fraud_flag = 1) AS false_negatives,
        countIf(rule_2_flag = 0 AND fraud_flag = 0) AS true_negatives,
        round(countIf(rule_2_flag = 1 AND fraud_flag = 1) * 100.0 / nullIf(countIf(rule_2_flag = 1), 0), 4) AS precision,
        round(countIf(rule_2_flag = 1 AND fraud_flag = 1) * 100.0 / nullIf(countIf(fraud_flag = 1), 0), 4) AS recall,
        round(2.0 * countIf(rule_2_flag = 1 AND fraud_flag = 1) / nullIf(countIf(rule_2_flag = 1) + countIf(fraud_flag = 1), 0) * 100, 4) AS f1_score,
        countIf(rule_2_flag = 1) AS total_flagged
    FROM rule_2_calc
    """,
    
    'rule_3': """
    WITH rule_3_calc AS (
        SELECT 
            trans_id,
            fraud_flag,
            CASE 
                WHEN COUNT(*) OVER (
                    PARTITION BY ac_from 
                    ORDER BY toUnixTimestamp(trans_initiate_time) ASC 
                    RANGE BETWEEN 3600 PRECEDING AND CURRENT ROW
                ) > 10 
                THEN 1 ELSE 0 
            END AS rule_3_flag
        FROM public.stixor_fraud_features_distributed
        WHERE cutoff_date BETWEEN '{start_date}' AND '{end_date}'
            AND mbar_account_type_name = 'Customer Account'
    )
    SELECT 
        'Rule 3: High Velocity' AS rule_name,
        countIf(rule_3_flag = 1 AND fraud_flag = 1) AS true_positives,
        countIf(rule_3_flag = 1 AND fraud_flag = 0) AS false_positives,
        countIf(rule_3_flag = 0 AND fraud_flag = 1) AS false_negatives,
        countIf(rule_3_flag = 0 AND fraud_flag = 0) AS true_negatives,
        round(countIf(rule_3_flag = 1 AND fraud_flag = 1) * 100.0 / nullIf(countIf(rule_3_flag = 1), 0), 4) AS precision,
        round(countIf(rule_3_flag = 1 AND fraud_flag = 1) * 100.0 / nullIf(countIf(fraud_flag = 1), 0), 4) AS recall,
        round(2.0 * countIf(rule_3_flag = 1 AND fraud_flag = 1) / nullIf(countIf(rule_3_flag = 1) + countIf(fraud_flag = 1), 0) * 100, 4) AS f1_score,
        countIf(rule_3_flag = 1) AS total_flagged
    FROM rule_3_calc
    """,
    
    'rule_4': """
    WITH rule_4_calc AS (
        SELECT 
            trans_id,
            fraud_flag,
            CASE 
                WHEN trx_amt >= 3 * MAX(trx_amt) OVER (
                    PARTITION BY ac_from 
                    ORDER BY toUnixTimestamp(trans_initiate_time)
                    RANGE BETWEEN 2592000 PRECEDING AND 1 PRECEDING
                ) 
                THEN 1 ELSE 0 
            END AS rule_4_flag
        FROM public.stixor_fraud_features_distributed
        WHERE cutoff_date BETWEEN '{start_date}' AND '{end_date}'
            AND mbar_account_type_name = 'Customer Account'
    )
    SELECT 
        'Rule 4: 3x Max Amount' AS rule_name,
        countIf(rule_4_flag = 1 AND fraud_flag = 1) AS true_positives,
        countIf(rule_4_flag = 1 AND fraud_flag = 0) AS false_positives,
        countIf(rule_4_flag = 0 AND fraud_flag = 1) AS false_negatives,
        countIf(rule_4_flag = 0 AND fraud_flag = 0) AS true_negatives,
        round(countIf(rule_4_flag = 1 AND fraud_flag = 1) * 100.0 / nullIf(countIf(rule_4_flag = 1), 0), 4) AS precision,
        round(countIf(rule_4_flag = 1 AND fraud_flag = 1) * 100.0 / nullIf(countIf(fraud_flag = 1), 0), 4) AS recall,
        round(2.0 * countIf(rule_4_flag = 1 AND fraud_flag = 1) / nullIf(countIf(rule_4_flag = 1) + countIf(fraud_flag = 1), 0) * 100, 4) AS f1_score,
        countIf(rule_4_flag = 1) AS total_flagged
    FROM rule_4_calc
    """,
    
    'rule_5': """
    WITH rule_5_calc AS (
        SELECT 
            trans_id,
            fraud_flag,
            CASE 
                WHEN uniqExact(trx_channel) OVER (
                    PARTITION BY ac_from 
                    ORDER BY toUnixTimestamp(trans_initiate_time)
                    RANGE BETWEEN 1800 PRECEDING AND CURRENT ROW
                ) > 1 
                THEN 1 ELSE 0 
            END AS rule_5_flag
        FROM public.stixor_fraud_features_distributed
        WHERE cutoff_date BETWEEN '{start_date}' AND '{end_date}'
            AND mbar_account_type_name = 'Customer Account'
    )
    SELECT 
        'Rule 5: Multiple Channels' AS rule_name,
        countIf(rule_5_flag = 1 AND fraud_flag = 1) AS true_positives,
        countIf(rule_5_flag = 1 AND fraud_flag = 0) AS false_positives,
        countIf(rule_5_flag = 0 AND fraud_flag = 1) AS false_negatives,
        countIf(rule_5_flag = 0 AND fraud_flag = 0) AS true_negatives,
        round(countIf(rule_5_flag = 1 AND fraud_flag = 1) * 100.0 / nullIf(countIf(rule_5_flag = 1), 0), 4) AS precision,
        round(countIf(rule_5_flag = 1 AND fraud_flag = 1) * 100.0 / nullIf(countIf(fraud_flag = 1), 0), 4) AS recall,
        round(2.0 * countIf(rule_5_flag = 1 AND fraud_flag = 1) / nullIf(countIf(rule_5_flag = 1) + countIf(fraud_flag = 1), 0) * 100, 4) AS f1_score,
        countIf(rule_5_flag = 1) AS total_flagged
    FROM rule_5_calc
    """,
    
    'rule_6': """
    WITH rule_6_calc AS (
        SELECT 
            trans_id,
            fraud_flag,
            CASE 
                WHEN toHour(trans_initiate_time) BETWEEN 1 AND 4 
                THEN 1 ELSE 0 
            END AS rule_6_flag
        FROM public.stixor_fraud_features_distributed
        WHERE cutoff_date BETWEEN '{start_date}' AND '{end_date}'
            AND mbar_account_type_name = 'Customer Account'
    )
    SELECT 
        'Rule 6: Off-Peak Hours' AS rule_name,
        countIf(rule_6_flag = 1 AND fraud_flag = 1) AS true_positives,
        countIf(rule_6_flag = 1 AND fraud_flag = 0) AS false_positives,
        countIf(rule_6_flag = 0 AND fraud_flag = 1) AS false_negatives,
        countIf(rule_6_flag = 0 AND fraud_flag = 0) AS true_negatives,
        round(countIf(rule_6_flag = 1 AND fraud_flag = 1) * 100.0 / nullIf(countIf(rule_6_flag = 1), 0), 4) AS precision,
        round(countIf(rule_6_flag = 1 AND fraud_flag = 1) * 100.0 / nullIf(countIf(fraud_flag = 1), 0), 4) AS recall,
        round(2.0 * countIf(rule_6_flag = 1 AND fraud_flag = 1) / nullIf(countIf(rule_6_flag = 1) + countIf(fraud_flag = 1), 0) * 100, 4) AS f1_score,
        countIf(rule_6_flag = 1) AS total_flagged
    FROM rule_6_calc
    """,
    
    'rule_time_weekend_night': """
    WITH rule_time_weekend_night_calc AS (
        SELECT 
            trans_id,
            fraud_flag,
            CASE 
                WHEN toDayOfWeek(trans_initiate_time) IN (6, 7)
                     AND toHour(trans_initiate_time) BETWEEN 22 AND 4
                THEN 1 ELSE 0 
            END AS rule_time_weekend_night_flag
        FROM public.stixor_fraud_features_distributed
        WHERE cutoff_date BETWEEN '{start_date}' AND '{end_date}'
          AND mbar_account_type_name = 'Customer Account'
    )
    SELECT 
        'Rule: Weekend Night Transaction' AS rule_name,
        countIf(rule_time_weekend_night_flag = 1 AND fraud_flag = 1) AS true_positives,
        countIf(rule_time_weekend_night_flag = 1 AND fraud_flag = 0) AS false_positives,
        countIf(rule_time_weekend_night_flag = 0 AND fraud_flag = 1) AS false_negatives,
        countIf(rule_time_weekend_night_flag = 0 AND fraud_flag = 0) AS true_negatives,
        round(countIf(rule_time_weekend_night_flag = 1 AND fraud_flag = 1) * 100.0 / nullIf(countIf(rule_time_weekend_night_flag = 1), 0), 4) AS precision,
        round(countIf(rule_time_weekend_night_flag = 1 AND fraud_flag = 1) * 100.0 / nullIf(countIf(fraud_flag = 1), 0), 4) AS recall,
        round(2.0 * countIf(rule_time_weekend_night_flag = 1 AND fraud_flag = 1) / nullIf(countIf(rule_time_weekend_night_flag = 1) + countIf(fraud_flag = 1), 0) * 100, 4) AS f1_score,
        countIf(rule_time_weekend_night_flag = 1) AS total_flagged
    FROM rule_time_weekend_night_calc
    """,
    
    'rule_time_unusual_hour': """
    WITH rule_time_unusual_hour_calc AS (
        SELECT 
            trans_id,
            fraud_flag,
            CASE 
                WHEN toHour(trans_initiate_time) IN (2, 3, 4)
                THEN 1 ELSE 0 
            END AS rule_time_unusual_hour_flag
        FROM public.stixor_fraud_features_distributed
        WHERE cutoff_date BETWEEN '{start_date}' AND '{end_date}'
          AND mbar_account_type_name = 'Customer Account'
    )
    SELECT 
        'Rule: Unusual Hour Transaction' AS rule_name,
        countIf(rule_time_unusual_hour_flag = 1 AND fraud_flag = 1) AS true_positives,
        countIf(rule_time_unusual_hour_flag = 1 AND fraud_flag = 0) AS false_positives,
        countIf(rule_time_unusual_hour_flag = 0 AND fraud_flag = 1) AS false_negatives,
        countIf(rule_time_unusual_hour_flag = 0 AND fraud_flag = 0) AS true_negatives,
        round(countIf(rule_time_unusual_hour_flag = 1 AND fraud_flag = 1) * 100.0 / nullIf(countIf(rule_time_unusual_hour_flag = 1), 0), 4) AS precision,
        round(countIf(rule_time_unusual_hour_flag = 1 AND fraud_flag = 1) * 100.0 / nullIf(countIf(fraud_flag = 1), 0), 4) AS recall,
        round(2.0 * countIf(rule_time_unusual_hour_flag = 1 AND fraud_flag = 1) / nullIf(countIf(rule_time_unusual_hour_flag = 1) + countIf(fraud_flag = 1), 0) * 100, 4) AS f1_score,
        countIf(rule_time_unusual_hour_flag = 1) AS total_flagged
    FROM rule_time_unusual_hour_calc
    """,
    
    'rule_age_new_high_amount': """
    WITH rule_age_new_high_amount_calc AS (
        SELECT 
            trans_id,
            fraud_flag,
            CASE 
                WHEN dateDiff('day', toDate(mbar_registered_date_time), cutoff_date) < 30 
                    AND trx_amt > 500 
                THEN 1 ELSE 0 
            END AS rule_age_new_high_amount_flag
        FROM public.stixor_fraud_features_distributed
        WHERE cutoff_date BETWEEN '{start_date}' AND '{end_date}'
          AND mbar_account_type_name = 'Customer Account'
    )
    SELECT 
        'Rule: New Account High Amount >$500' AS rule_name,
        countIf(rule_age_new_high_amount_flag = 1 AND fraud_flag = 1) AS true_positives,
        countIf(rule_age_new_high_amount_flag = 1 AND fraud_flag = 0) AS false_positives,
        countIf(rule_age_new_high_amount_flag = 0 AND fraud_flag = 1) AS false_negatives,
        countIf(rule_age_new_high_amount_flag = 0 AND fraud_flag = 0) AS true_negatives,
        round(countIf(rule_age_new_high_amount_flag = 1 AND fraud_flag = 1) * 100.0 / nullIf(countIf(rule_age_new_high_amount_flag = 1), 0), 4) AS precision,
        round(countIf(rule_age_new_high_amount_flag = 1 AND fraud_flag = 1) * 100.0 / nullIf(countIf(fraud_flag = 1), 0), 4) AS recall,
        round(2.0 * countIf(rule_age_new_high_amount_flag = 1 AND fraud_flag = 1) / nullIf(countIf(rule_age_new_high_amount_flag = 1) + countIf(fraud_flag = 1), 0) * 100, 4) AS f1_score,
        countIf(rule_age_new_high_amount_flag = 1) AS total_flagged
    FROM rule_age_new_high_amount_calc
    """,
    
    'rule_age_very_new_any_txn': """
    WITH rule_age_very_new_any_txn_calc AS (
        SELECT 
            trans_id,
            fraud_flag,
            CASE 
                WHEN dateDiff('day', toDate(mbar_registered_date_time), cutoff_date) < 7 
                THEN 1 ELSE 0 
            END AS rule_age_very_new_any_txn_flag
        FROM public.stixor_fraud_features_distributed
        WHERE cutoff_date BETWEEN '{start_date}' AND '{end_date}'
          AND mbar_account_type_name = 'Customer Account'
    )
    SELECT 
        'Rule: Very New Account <7 days Any Txn' AS rule_name,
        countIf(rule_age_very_new_any_txn_flag = 1 AND fraud_flag = 1) AS true_positives,
        countIf(rule_age_very_new_any_txn_flag = 1 AND fraud_flag = 0) AS false_positives,
        countIf(rule_age_very_new_any_txn_flag = 0 AND fraud_flag = 1) AS false_negatives,
        countIf(rule_age_very_new_any_txn_flag = 0 AND fraud_flag = 0) AS true_negatives,
        round(countIf(rule_age_very_new_any_txn_flag = 1 AND fraud_flag = 1) * 100.0 / nullIf(countIf(rule_age_very_new_any_txn_flag = 1), 0), 4) AS precision,
        round(countIf(rule_age_very_new_any_txn_flag = 1 AND fraud_flag = 1) * 100.0 / nullIf(countIf(fraud_flag = 1), 0), 4) AS recall,
        round(2.0 * countIf(rule_age_very_new_any_txn_flag = 1 AND fraud_flag = 1) / nullIf(countIf(rule_age_very_new_any_txn_flag = 1) + countIf(fraud_flag = 1), 0) * 100, 4) AS f1_score,
        countIf(rule_age_very_new_any_txn_flag = 1) AS total_flagged
    FROM rule_age_very_new_any_txn_calc
    """,
    
    'rule_age_immediate_activity': """
    WITH rule_age_immediate_activity_calc AS (
        SELECT 
            trans_id,
            fraud_flag,
            CASE 
                WHEN dateDiff('hour', mbar_registered_date_time, trans_initiate_time) < 1 
                THEN 1 ELSE 0 
            END AS rule_age_immediate_activity_flag
        FROM public.stixor_fraud_features_distributed
        WHERE cutoff_date BETWEEN '{start_date}' AND '{end_date}'
          AND mbar_account_type_name = 'Customer Account'
    )
    SELECT 
        'Rule: First Txn <1hr after Registration' AS rule_name,
        countIf(rule_age_immediate_activity_flag = 1 AND fraud_flag = 1) AS true_positives,
        countIf(rule_age_immediate_activity_flag = 1 AND fraud_flag = 0) AS false_positives,
        countIf(rule_age_immediate_activity_flag = 0 AND fraud_flag = 1) AS false_negatives,
        countIf(rule_age_immediate_activity_flag = 0 AND fraud_flag = 0) AS true_negatives,
        round(countIf(rule_age_immediate_activity_flag = 1 AND fraud_flag = 1) * 100.0 / nullIf(countIf(rule_age_immediate_activity_flag = 1), 0), 4) AS precision,
        round(countIf(rule_age_immediate_activity_flag = 1 AND fraud_flag = 1) * 100.0 / nullIf(countIf(fraud_flag = 1), 0), 4) AS recall,
        round(2.0 * countIf(rule_age_immediate_activity_flag = 1 AND fraud_flag = 1) / nullIf(countIf(rule_age_immediate_activity_flag = 1) + countIf(fraud_flag = 1), 0) * 100, 4) AS f1_score,
        countIf(rule_age_immediate_activity_flag = 1) AS total_flagged
    FROM rule_age_immediate_activity_calc
    """,
    
    'rule_age_dormant_30_high_amount': """
    WITH rule_age_dormant_30_high_amount_calc AS (
        SELECT 
            trans_id,
            fraud_flag,
            CASE 
                WHEN dateDiff('day', 
                    LAG(trans_initiate_time, 1) OVER (PARTITION BY ac_from ORDER BY trans_initiate_time),
                    trans_initiate_time
                ) > 30 
                AND trx_amt > 1000
                THEN 1 ELSE 0 
            END AS rule_age_dormant_30_high_amount_flag
        FROM public.stixor_fraud_features_distributed
        WHERE cutoff_date BETWEEN '{start_date}' AND '{end_date}'
          AND mbar_account_type_name = 'Customer Account'
    )
    SELECT 
        'Rule: Dormant >30d, High txn' AS rule_name,
        countIf(rule_age_dormant_30_high_amount_flag = 1 AND fraud_flag = 1) AS true_positives,
        countIf(rule_age_dormant_30_high_amount_flag = 1 AND fraud_flag = 0) AS false_positives,
        countIf(rule_age_dormant_30_high_amount_flag = 0 AND fraud_flag = 1) AS false_negatives,
        countIf(rule_age_dormant_30_high_amount_flag = 0 AND fraud_flag = 0) AS true_negatives,
        round(countIf(rule_age_dormant_30_high_amount_flag = 1 AND fraud_flag = 1) * 100.0 / nullIf(countIf(rule_age_dormant_30_high_amount_flag = 1), 0), 4) AS precision,
        round(countIf(rule_age_dormant_30_high_amount_flag = 1 AND fraud_flag = 1) * 100.0 / nullIf(countIf(fraud_flag = 1), 0), 4) AS recall,
        round(2.0 * countIf(rule_age_dormant_30_high_amount_flag = 1 AND fraud_flag = 1) / nullIf(countIf(rule_age_dormant_30_high_amount_flag = 1) + countIf(fraud_flag = 1), 0) * 100, 4) AS f1_score,
        countIf(rule_age_dormant_30_high_amount_flag = 1) AS total_flagged
    FROM rule_age_dormant_30_high_amount_calc
    """,
    
    'rule_amount_high_value': """
    WITH rule_amount_high_value_calc AS (
        SELECT 
            trans_id,
            fraud_flag,
            CASE 
                WHEN trx_amt > 5000 
                THEN 1 ELSE 0 
            END AS rule_amount_high_value_flag
        FROM public.stixor_fraud_features_distributed
        WHERE cutoff_date BETWEEN '{start_date}' AND '{end_date}'
          AND mbar_account_type_name = 'Customer Account'
    )
    SELECT 
        'Rule: High Value txn >$5k' AS rule_name,
        countIf(rule_amount_high_value_flag = 1 AND fraud_flag = 1) AS true_positives,
        countIf(rule_amount_high_value_flag = 1 AND fraud_flag = 0) AS false_positives,
        countIf(rule_amount_high_value_flag = 0 AND fraud_flag = 1) AS false_negatives,
        countIf(rule_amount_high_value_flag = 0 AND fraud_flag = 0) AS true_negatives,
        round(countIf(rule_amount_high_value_flag = 1 AND fraud_flag = 1) * 100.0 / nullIf(countIf(rule_amount_high_value_flag = 1), 0), 4) AS precision,
        round(countIf(rule_amount_high_value_flag = 1 AND fraud_flag = 1) * 100.0 / nullIf(countIf(fraud_flag = 1), 0), 4) AS recall,
        round(2.0 * countIf(rule_amount_high_value_flag = 1 AND fraud_flag = 1) / nullIf(countIf(rule_amount_high_value_flag = 1) + countIf(fraud_flag = 1), 0) * 100, 4) AS f1_score,
        countIf(rule_amount_high_value_flag = 1) AS total_flagged
    FROM rule_amount_high_value_calc
    """,
    
    'rule_amount_round_number': """
    WITH rule_amount_round_number_calc AS (
        SELECT 
            trans_id,
            fraud_flag,
            CASE 
                WHEN trx_amt % 1000 = 0 AND trx_amt > 1000
                THEN 1 ELSE 0 
            END AS rule_amount_round_number_flag
        FROM public.stixor_fraud_features_distributed
        WHERE cutoff_date BETWEEN '{start_date}' AND '{end_date}'
          AND mbar_account_type_name = 'Customer Account'
    )
    SELECT 
        'Rule: Round Number txn' AS rule_name,
        countIf(rule_amount_round_number_flag = 1 AND fraud_flag = 1) AS true_positives,
        countIf(rule_amount_round_number_flag = 1 AND fraud_flag = 0) AS false_positives,
        countIf(rule_amount_round_number_flag = 0 AND fraud_flag = 1) AS false_negatives,
        countIf(rule_amount_round_number_flag = 0 AND fraud_flag = 0) AS true_negatives,
        round(countIf(rule_amount_round_number_flag = 1 AND fraud_flag = 1) * 100.0 / nullIf(countIf(rule_amount_round_number_flag = 1), 0), 4) AS precision,
        round(countIf(rule_amount_round_number_flag = 1 AND fraud_flag = 1) * 100.0 / nullIf(countIf(fraud_flag = 1), 0), 4) AS recall,
        round(2.0 * countIf(rule_amount_round_number_flag = 1 AND fraud_flag = 1) / nullIf(countIf(rule_amount_round_number_flag = 1) + countIf(fraud_flag = 1), 0) * 100, 4) AS f1_score,
        countIf(rule_amount_round_number_flag = 1) AS total_flagged
    FROM rule_amount_round_number_calc
    """,
    
    'rule_amount_structuring': """
    WITH rule_amount_structuring_calc AS (
        SELECT 
            trans_id,
            fraud_flag,
            CASE 
                WHEN (trx_amt BETWEEN 4900 AND 4999) OR (trx_amt BETWEEN 9900 AND 9999)
                    THEN 1 ELSE 0
            END AS rule_amount_structuring_flag
        FROM public.stixor_fraud_features_distributed
        WHERE cutoff_date BETWEEN '{start_date}' AND '{end_date}'
          AND mbar_account_type_name = 'Customer Account'
    )
    SELECT 
        'Rule: Structuring (Just-below thresh)' AS rule_name,
        countIf(rule_amount_structuring_flag = 1 AND fraud_flag = 1) AS true_positives,
        countIf(rule_amount_structuring_flag = 1 AND fraud_flag = 0) AS false_positives,
        countIf(rule_amount_structuring_flag = 0 AND fraud_flag = 1) AS false_negatives,
        countIf(rule_amount_structuring_flag = 0 AND fraud_flag = 0) AS true_negatives,
        round(countIf(rule_amount_structuring_flag = 1 AND fraud_flag = 1) * 100.0 / nullIf(countIf(rule_amount_structuring_flag = 1), 0), 4) AS precision,
        round(countIf(rule_amount_structuring_flag = 1 AND fraud_flag = 1) * 100.0 / nullIf(countIf(fraud_flag = 1), 0), 4) AS recall,
        round(2.0 * countIf(rule_amount_structuring_flag = 1 AND fraud_flag = 1) / nullIf(countIf(rule_amount_structuring_flag = 1) + countIf(fraud_flag = 1), 0) * 100, 4) AS f1_score,
        countIf(rule_amount_structuring_flag = 1) AS total_flagged
    FROM rule_amount_structuring_calc
    """,
    
    'rule_amount_spike': """
    WITH rule_amount_spike_calc AS (
        SELECT 
            trans_id,
            fraud_flag,
            trx_amt,
            avg(trx_amt) OVER (
                PARTITION BY ac_from
                ORDER BY trans_initiate_time
                ROWS BETWEEN 100 PRECEDING AND 1 PRECEDING
            ) AS avg_amt,
            CASE 
                WHEN trx_amt > 3 * avg(trx_amt) OVER (
                    PARTITION BY ac_from
                    ORDER BY trans_initiate_time
                    ROWS BETWEEN 100 PRECEDING AND 1 PRECEDING
                )
                    THEN 1 ELSE 0
            END AS rule_amount_spike_flag
        FROM public.stixor_fraud_features_distributed
        WHERE cutoff_date BETWEEN '{start_date}' AND '{end_date}'
          AND mbar_account_type_name = 'Customer Account'
    )
    SELECT 
        'Rule: Amount Spike >3x AVG' AS rule_name,
        countIf(rule_amount_spike_flag = 1 AND fraud_flag = 1) AS true_positives,
        countIf(rule_amount_spike_flag = 1 AND fraud_flag = 0) AS false_positives,
        countIf(rule_amount_spike_flag = 0 AND fraud_flag = 1) AS false_negatives,
        countIf(rule_amount_spike_flag = 0 AND fraud_flag = 0) AS true_negatives,
        round(countIf(rule_amount_spike_flag = 1 AND fraud_flag = 1) * 100.0 / nullIf(countIf(rule_amount_spike_flag = 1), 0), 4) AS precision,
        round(countIf(rule_amount_spike_flag = 1 AND fraud_flag = 1) * 100.0 / nullIf(countIf(fraud_flag = 1), 0), 4) AS recall,
        round(2.0 * countIf(rule_amount_spike_flag = 1 AND fraud_flag = 1) / nullIf(countIf(rule_amount_spike_flag = 1) + countIf(fraud_flag = 1), 0) * 100, 4) AS f1_score,
        countIf(rule_amount_spike_flag = 1) AS total_flagged
    FROM rule_amount_spike_calc
    """,
    
    'rule_amount_3_stddev': """
    WITH rule_amount_3_stddev_calc AS (
        SELECT 
            trans_id,
            fraud_flag,
            trx_amt,
            avg(trx_amt) OVER (
                PARTITION BY ac_from
                ORDER BY trans_initiate_time
                ROWS BETWEEN 100 PRECEDING AND 1 PRECEDING
            ) AS avg_amt,
            stddevPop(trx_amt) OVER (
                PARTITION BY ac_from
                ORDER BY trans_initiate_time
                ROWS BETWEEN 100 PRECEDING AND 1 PRECEDING
            ) AS stddev_amt,
            CASE 
                WHEN trx_amt > avg(trx_amt) OVER (
                    PARTITION BY ac_from
                    ORDER BY trans_initiate_time
                    ROWS BETWEEN 100 PRECEDING AND 1 PRECEDING
                ) + 3 * stddevPop(trx_amt) OVER (
                    PARTITION BY ac_from
                    ORDER BY trans_initiate_time
                    ROWS BETWEEN 100 PRECEDING AND 1 PRECEDING
                )
                    THEN 1 ELSE 0
            END AS rule_amount_3_stddev_flag
        FROM public.stixor_fraud_features_distributed
        WHERE cutoff_date BETWEEN '{start_date}' AND '{end_date}'
          AND mbar_account_type_name = 'Customer Account'
    )
    SELECT 
        'Rule: Amount >3 StdDev' AS rule_name,
        countIf(rule_amount_3_stddev_flag = 1 AND fraud_flag = 1) AS true_positives,
        countIf(rule_amount_3_stddev_flag = 1 AND fraud_flag = 0) AS false_positives,
        countIf(rule_amount_3_stddev_flag = 0 AND fraud_flag = 1) AS false_negatives,
        countIf(rule_amount_3_stddev_flag = 0 AND fraud_flag = 0) AS true_negatives,
        round(countIf(rule_amount_3_stddev_flag = 1 AND fraud_flag = 1) * 100.0 / nullIf(countIf(rule_amount_3_stddev_flag = 1), 0), 4) AS precision,
        round(countIf(rule_amount_3_stddev_flag = 1 AND fraud_flag = 1) * 100.0 / nullIf(countIf(fraud_flag = 1), 0), 4) AS recall,
        round(2.0 * countIf(rule_amount_3_stddev_flag = 1 AND fraud_flag = 1) / nullIf(countIf(rule_amount_3_stddev_flag = 1) + countIf(fraud_flag = 1), 0) * 100, 4) AS f1_score,
        countIf(rule_amount_3_stddev_flag = 1) AS total_flagged
    FROM rule_amount_3_stddev_calc
    """,
    
    'rule_channel_pgw_new_high': """
    WITH rule_channel_pgw_new_high_calc AS (
        SELECT 
            trans_id,
            fraud_flag,
            CASE 
                WHEN trx_channel IN ('Payment Gateway', 'PGW')
                     AND dateDiff('day', toDate(mbar_registered_date_time), cutoff_date) < 30
                     AND trx_amt > 1000
                    THEN 1 ELSE 0
            END AS rule_channel_pgw_new_high_flag
        FROM public.stixor_fraud_features_distributed
        WHERE cutoff_date BETWEEN '{start_date}' AND '{end_date}'
          AND mbar_account_type_name = 'Customer Account'
    )
    SELECT 
        'Rule: PGW NewAcct HighAmt' AS rule_name,
        countIf(rule_channel_pgw_new_high_flag = 1 AND fraud_flag = 1) AS true_positives,
        countIf(rule_channel_pgw_new_high_flag = 1 AND fraud_flag = 0) AS false_positives,
        countIf(rule_channel_pgw_new_high_flag = 0 AND fraud_flag = 1) AS false_negatives,
        countIf(rule_channel_pgw_new_high_flag = 0 AND fraud_flag = 0) AS true_negatives,
        round(countIf(rule_channel_pgw_new_high_flag = 1 AND fraud_flag = 1) * 100.0 / nullIf(countIf(rule_channel_pgw_new_high_flag = 1), 0), 4) AS precision,
        round(countIf(rule_channel_pgw_new_high_flag = 1 AND fraud_flag = 1) * 100.0 / nullIf(countIf(fraud_flag = 1), 0), 4) AS recall,
        round(2.0 * countIf(rule_channel_pgw_new_high_flag = 1 AND fraud_flag = 1) / nullIf(countIf(rule_channel_pgw_new_high_flag = 1) + countIf(fraud_flag = 1), 0) * 100, 4) AS f1_score,
        countIf(rule_channel_pgw_new_high_flag = 1) AS total_flagged
    FROM rule_channel_pgw_new_high_calc
    """,
    
    'rule_channel_mobile_new_high': """
    WITH rule_channel_mobile_new_high_calc AS (
        SELECT 
            trans_id,
            fraud_flag,
            CASE 
                WHEN trx_channel IN ('Mobile App', 'NEW_JC_APP')
                     AND dateDiff('day', toDate(mbar_registered_date_time), cutoff_date) < 30
                     AND trx_amt > 1000
                    THEN 1 ELSE 0
            END AS rule_channel_mobile_new_high_flag
        FROM public.stixor_fraud_features_distributed
        WHERE cutoff_date BETWEEN '{start_date}' AND '{end_date}'
          AND mbar_account_type_name = 'Customer Account'
    )
    SELECT 
        'Rule: Mobile NewAcct HighAmt' AS rule_name,
        countIf(rule_channel_mobile_new_high_flag = 1 AND fraud_flag = 1) AS true_positives,
        countIf(rule_channel_mobile_new_high_flag = 1 AND fraud_flag = 0) AS false_positives,
        countIf(rule_channel_mobile_new_high_flag = 0 AND fraud_flag = 1) AS false_negatives,
        countIf(rule_channel_mobile_new_high_flag = 0 AND fraud_flag = 0) AS true_negatives,
        round(countIf(rule_channel_mobile_new_high_flag = 1 AND fraud_flag = 1) * 100.0 / nullIf(countIf(rule_channel_mobile_new_high_flag = 1), 0), 4) AS precision,
        round(countIf(rule_channel_mobile_new_high_flag = 1 AND fraud_flag = 1) * 100.0 / nullIf(countIf(fraud_flag = 1), 0), 4) AS recall,
        round(2.0 * countIf(rule_channel_mobile_new_high_flag = 1 AND fraud_flag = 1) / nullIf(countIf(rule_channel_mobile_new_high_flag = 1) + countIf(fraud_flag = 1), 0) * 100, 4) AS f1_score,
        countIf(rule_channel_mobile_new_high_flag = 1) AS total_flagged
    FROM rule_channel_mobile_new_high_calc
    """,
    
    'rule_behavior_new_recipient_high': """
    WITH rule_behavior_new_recipient_high_calc AS (
        SELECT 
            trans_id,
            fraud_flag,
            ROW_NUMBER() OVER (PARTITION BY ac_from, ac_to ORDER BY trans_initiate_time) AS rn,
            trx_amt,
            CASE 
                WHEN ROW_NUMBER() OVER (PARTITION BY ac_from, ac_to ORDER BY trans_initiate_time) = 1 AND trx_amt > 1000 
                THEN 1 ELSE 0
            END AS rule_behavior_new_recipient_high_flag
        FROM public.stixor_fraud_features_distributed
        WHERE cutoff_date BETWEEN '{start_date}' AND '{end_date}'
          AND mbar_account_type_name = 'Customer Account'
    )
    SELECT 
        'Rule: New Recipient HighAmt' AS rule_name,
        countIf(rule_behavior_new_recipient_high_flag = 1 AND fraud_flag = 1) AS true_positives,
        countIf(rule_behavior_new_recipient_high_flag = 1 AND fraud_flag = 0) AS false_positives,
        countIf(rule_behavior_new_recipient_high_flag = 0 AND fraud_flag = 1) AS false_negatives,
        countIf(rule_behavior_new_recipient_high_flag = 0 AND fraud_flag = 0) AS true_negatives,
        round(countIf(rule_behavior_new_recipient_high_flag = 1 AND fraud_flag = 1) * 100.0 / nullIf(countIf(rule_behavior_new_recipient_high_flag = 1), 0), 4) AS precision,
        round(countIf(rule_behavior_new_recipient_high_flag = 1 AND fraud_flag = 1) * 100.0 / nullIf(countIf(fraud_flag = 1), 0), 4) AS recall,
        round(2.0 * countIf(rule_behavior_new_recipient_high_flag = 1 AND fraud_flag = 1) / nullIf(countIf(rule_behavior_new_recipient_high_flag = 1) + countIf(fraud_flag = 1), 0) * 100, 4) AS f1_score,
        countIf(rule_behavior_new_recipient_high_flag = 1) AS total_flagged
    FROM rule_behavior_new_recipient_high_calc
    """,
    
    'rule_behavior_amount_jump': """
    WITH rule_behavior_amount_jump_calc AS (
        SELECT 
            trans_id,
            fraud_flag,
            trx_amt,
            avg(trx_amt) OVER (
                PARTITION BY ac_from
                ORDER BY trans_initiate_time
                ROWS BETWEEN 50 PRECEDING AND 1 PRECEDING
            ) AS avg_amt,
            CASE 
                WHEN trx_amt > 5 * avg(trx_amt) OVER (
                    PARTITION BY ac_from
                    ORDER BY trans_initiate_time
                    ROWS BETWEEN 50 PRECEDING AND 1 PRECEDING
                )
                     AND avg(trx_amt) OVER (
                    PARTITION BY ac_from
                    ORDER BY trans_initiate_time
                    ROWS BETWEEN 50 PRECEDING AND 1 PRECEDING
                ) < 1000
                    THEN 1 ELSE 0
            END AS rule_behavior_amount_jump_flag
        FROM public.stixor_fraud_features_distributed
        WHERE cutoff_date BETWEEN '{start_date}' AND '{end_date}'
          AND mbar_account_type_name = 'Customer Account'
    )
    SELECT 
        'Rule: Amount Jump 5x AVG, AVG<1k' AS rule_name,
        countIf(rule_behavior_amount_jump_flag = 1 AND fraud_flag = 1) AS true_positives,
        countIf(rule_behavior_amount_jump_flag = 1 AND fraud_flag = 0) AS false_positives,
        countIf(rule_behavior_amount_jump_flag = 0 AND fraud_flag = 1) AS false_negatives,
        countIf(rule_behavior_amount_jump_flag = 0 AND fraud_flag = 0) AS true_negatives,
        round(countIf(rule_behavior_amount_jump_flag = 1 AND fraud_flag = 1) * 100.0 / nullIf(countIf(rule_behavior_amount_jump_flag = 1), 0), 4) AS precision,
        round(countIf(rule_behavior_amount_jump_flag = 1 AND fraud_flag = 1) * 100.0 / nullIf(countIf(fraud_flag = 1), 0), 4) AS recall,
        round(2.0 * countIf(rule_behavior_amount_jump_flag = 1 AND fraud_flag = 1) / nullIf(countIf(rule_behavior_amount_jump_flag = 1) + countIf(fraud_flag = 1), 0) * 100, 4) AS f1_score,
        countIf(rule_behavior_amount_jump_flag = 1) AS total_flagged
    FROM rule_behavior_amount_jump_calc
    """,
    
    'rule_recipient_money_mule': """
    WITH rule_recipient_money_mule_calc AS (
        SELECT 
            trans_id,
            fraud_flag,
            count(DISTINCT ac_from) OVER (
                PARTITION BY ac_to
                ORDER BY toUnixTimestamp(trans_initiate_time)
                RANGE BETWEEN 86400 PRECEDING AND CURRENT ROW
            ) AS num_senders_24h,
            CASE 
                WHEN count(DISTINCT ac_from) OVER (
                    PARTITION BY ac_to
                    ORDER BY toUnixTimestamp(trans_initiate_time)
                    RANGE BETWEEN 86400 PRECEDING AND CURRENT ROW
                ) > 10 
                THEN 1 ELSE 0
            END AS rule_recipient_money_mule_flag
        FROM public.stixor_fraud_features_distributed
        WHERE cutoff_date BETWEEN '{start_date}' AND '{end_date}'
          AND mbar_account_type_name = 'Customer Account'
    )
    SELECT 
        'Rule: Money Mule >10 In 24h' AS rule_name,
        countIf(rule_recipient_money_mule_flag = 1 AND fraud_flag = 1) AS true_positives,
        countIf(rule_recipient_money_mule_flag = 1 AND fraud_flag = 0) AS false_positives,
        countIf(rule_recipient_money_mule_flag = 0 AND fraud_flag = 1) AS false_negatives,
        countIf(rule_recipient_money_mule_flag = 0 AND fraud_flag = 0) AS true_negatives,
        round(countIf(rule_recipient_money_mule_flag = 1 AND fraud_flag = 1) * 100.0 / nullIf(countIf(rule_recipient_money_mule_flag = 1), 0), 4) AS precision,
        round(countIf(rule_recipient_money_mule_flag = 1 AND fraud_flag = 1) * 100.0 / nullIf(countIf(fraud_flag = 1), 0), 4) AS recall,
        round(2.0 * countIf(rule_recipient_money_mule_flag = 1 AND fraud_flag = 1) / nullIf(countIf(rule_recipient_money_mule_flag = 1) + countIf(fraud_flag = 1), 0) * 100, 4) AS f1_score,
        countIf(rule_recipient_money_mule_flag = 1) AS total_flagged
    FROM rule_recipient_money_mule_calc
    """,
    
    'rule_recipient_high_amount_1hour': """
    WITH rule_recipient_high_amount_1hour_calc AS (
        SELECT 
            trans_id,
            fraud_flag,
            sum(trx_amt) OVER (
                PARTITION BY ac_to
                ORDER BY toUnixTimestamp(trans_initiate_time)
                RANGE BETWEEN 3600 PRECEDING AND CURRENT ROW
            ) AS sum_amt_1h,
            CASE 
                WHEN sum(trx_amt) OVER (
                    PARTITION BY ac_to
                    ORDER BY toUnixTimestamp(trans_initiate_time)
                    RANGE BETWEEN 3600 PRECEDING AND CURRENT ROW
                ) > 10000 
                THEN 1 ELSE 0
            END AS rule_recipient_high_amount_1hour_flag
        FROM public.stixor_fraud_features_distributed
        WHERE cutoff_date BETWEEN '{start_date}' AND '{end_date}'
          AND mbar_account_type_name = 'Customer Account'
    )
    SELECT 
        'Rule: Recipient >$10k In 1hr' AS rule_name,
        countIf(rule_recipient_high_amount_1hour_flag = 1 AND fraud_flag = 1) AS true_positives,
        countIf(rule_recipient_high_amount_1hour_flag = 1 AND fraud_flag = 0) AS false_positives,
        countIf(rule_recipient_high_amount_1hour_flag = 0 AND fraud_flag = 1) AS false_negatives,
        countIf(rule_recipient_high_amount_1hour_flag = 0 AND fraud_flag = 0) AS true_negatives,
        round(countIf(rule_recipient_high_amount_1hour_flag = 1 AND fraud_flag = 1) * 100.0 / nullIf(countIf(rule_recipient_high_amount_1hour_flag = 1), 0), 4) AS precision,
        round(countIf(rule_recipient_high_amount_1hour_flag = 1 AND fraud_flag = 1) * 100.0 / nullIf(countIf(fraud_flag = 1), 0), 4) AS recall,
        round(2.0 * countIf(rule_recipient_high_amount_1hour_flag = 1 AND fraud_flag = 1) / nullIf(countIf(rule_recipient_high_amount_1hour_flag = 1) + countIf(fraud_flag = 1), 0) * 100, 4) AS f1_score,
        countIf(rule_recipient_high_amount_1hour_flag = 1) AS total_flagged
    FROM rule_recipient_high_amount_1hour_calc
    """
}

print(f"✅ Defined {len(rule_queries)} fraud detection rules")
print("\n📋 Rule Categories:")
print("   Base Rules (1-6): 6 rules")
print("   Time-based Rules: 2 rules")
print("   Account Age Rules: 4 rules") 
print("   Amount Rules: 6 rules")
print("   Channel Rules: 2 rules")
print("   Behavioral Rules: 2 rules")
print("   Recipient Rules: 2 rules")
print(f"\n   TOTAL: {len(rule_queries)} rules")

print("\n📝 Rule List:")
for i, rule_name in enumerate(rule_queries.keys(), 1):
    print(f"   {i}. {rule_name}")


✅ Defined 23 fraud detection rules

📋 Rule Categories:
   Base Rules (1-6): 6 rules
   Time-based Rules: 2 rules
   Account Age Rules: 4 rules
   Amount Rules: 6 rules
   Channel Rules: 2 rules
   Behavioral Rules: 2 rules
   Recipient Rules: 2 rules

   TOTAL: 23 rules

📝 Rule List:
   1. rule_1
   2. rule_2
   3. rule_3
   4. rule_4
   5. rule_5
   6. rule_6
   7. rule_time_weekend_night
   8. rule_time_unusual_hour
   9. rule_age_new_high_amount
   10. rule_age_very_new_any_txn
   11. rule_age_immediate_activity
   12. rule_age_dormant_30_high_amount
   13. rule_amount_high_value
   14. rule_amount_round_number
   15. rule_amount_structuring
   16. rule_amount_spike
   17. rule_amount_3_stddev
   18. rule_channel_pgw_new_high
   19. rule_channel_mobile_new_high
   20. rule_behavior_new_recipient_high
   21. rule_behavior_amount_jump
   22. rule_recipient_money_mule
   23. rule_recipient_high_amount_1hour


## 5. Execute All Rules and Collect Results

In [6]:
import time

# List to store results
all_results = []

print("🚀 Executing fraud detection rules in ClickHouse...\n")
print("="*80)

for rule_name, query_template in rule_queries.items():
    print(f"\n📊 Executing {rule_name.upper()}...")
    
    # Format query with date parameters
    query = query_template.format(start_date=START_DATE, end_date=END_DATE)
    
    try:
        start_time = time.time()
        
        # Execute query
        result = client.execute(query)
        
        execution_time = time.time() - start_time
        
        # Extract results
        if result:
            row = result[0]
            rule_data = {
                'rule_id': rule_name,
                'rule_name': row[0],
                'true_positives': row[1],
                'false_positives': row[2],
                'false_negatives': row[3],
                'true_negatives': row[4],
                'precision': row[5],
                'recall': row[6],
                'f1_score': row[7],
                'total_flagged': row[8],
                'execution_time_sec': round(execution_time, 2)
            }
            all_results.append(rule_data)
            
            print(f"   ✅ Completed in {execution_time:.2f} seconds")
            print(f"   • Precision: {row[5]}%")
            print(f"   • Recall: {row[6]}%")
            print(f"   • F1 Score: {row[7]}")
            print(f"   • Total Flagged: {row[8]:,}")
        else:
            print(f"   ⚠️ No results returned")
            
    except Exception as e:
        print(f"   ❌ Error: {str(e)}")
        continue

print("\n" + "="*80)
print(f"\n✅ Completed execution of {len(all_results)} rules")

🚀 Executing fraud detection rules in ClickHouse...


📊 Executing RULE_1...
   ✅ Completed in 1.10 seconds
   • Precision: 0.0165%
   • Recall: 1.193%
   • F1 Score: 0.0326
   • Total Flagged: 381,618

📊 Executing RULE_2...
   ✅ Completed in 1.17 seconds
   • Precision: 0.004%
   • Recall: 0.0189%
   • F1 Score: 0.0066
   • Total Flagged: 25,176

📊 Executing RULE_3...
   ✅ Completed in 52.35 seconds
   • Precision: 0.0179%
   • Recall: 12.1757%
   • F1 Score: 0.0357
   • Total Flagged: 3,593,481

📊 Executing RULE_4...
   ✅ Completed in 55.82 seconds
   • Precision: 0.0015%
   • Recall: 9.2407%
   • F1 Score: 0.003
   • Total Flagged: 32,684,194

📊 Executing RULE_5...
   ✅ Completed in 66.10 seconds
   • Precision: 0.0074%
   • Recall: 18.6707%
   • F1 Score: 0.0147
   • Total Flagged: 13,390,487

📊 Executing RULE_6...
   ✅ Completed in 0.65 seconds
   • Precision: 0.0009%
   • Recall: 2.8404%
   • F1 Score: 0.0018
   • Total Flagged: 16,602,870

📊 Executing RULE_TIME_WEEKEND_NIGHT...
  

## 6. Create DataFrame with All Results

In [7]:
# Create DataFrame from results
rules_df = pd.DataFrame(all_results)

# Display results
print("📊 FRAUD RULES PERFORMANCE SUMMARY")
print("="*100)
print(rules_df.to_string(index=False))
print("="*100)

📊 FRAUD RULES PERFORMANCE SUMMARY
                         rule_id                               rule_name  true_positives  false_positives  false_negatives  true_negatives  precision  recall  f1_score  total_flagged  execution_time_sec
                          rule_1         Rule 1: New Account High Amount              63           381555             5218       280197916     0.0165  1.1930    0.0326         381618                1.10
                          rule_2                Rule 2: Very New Account               1            25175             5280       280554296     0.0040  0.0189    0.0066          25176                1.17
                          rule_3                   Rule 3: High Velocity             643          3592838             4638       276986633     0.0179 12.1757    0.0357        3593481               52.35
                          rule_4                   Rule 4: 3x Max Amount             488         32683706             4793       247895765     0.0015  9.2

## 7. Analyze Results

In [8]:
# Calculate FPR (False Positive Rate)
rules_df['fpr'] = (rules_df['false_positives'] / 
                   (rules_df['false_positives'] + rules_df['true_negatives']) * 100).round(4)

# Calculate Accuracy
rules_df['accuracy'] = ((rules_df['true_positives'] + rules_df['true_negatives']) / 
                        (rules_df['true_positives'] + rules_df['false_positives'] + 
                         rules_df['false_negatives'] + rules_df['true_negatives']) * 100).round(4)

print("\n📈 ADDITIONAL METRICS")
print("="*100)
print(rules_df[['rule_name', 'precision', 'recall', 'f1_score', 'fpr', 'accuracy']].to_string(index=False))
print("="*100)


📈 ADDITIONAL METRICS
                              rule_name  precision  recall  f1_score     fpr  accuracy
        Rule 1: New Account High Amount     0.0165  1.1930    0.0326  0.1360   99.8622
               Rule 2: Very New Account     0.0040  0.0189    0.0066  0.0090   99.9891
                  Rule 3: High Velocity     0.0179 12.1757    0.0357  1.2805   98.7179
                  Rule 4: 3x Max Amount     0.0015  9.2407    0.0030 11.6486   88.3499
              Rule 5: Multiple Channels     0.0074 18.6707    0.0147  4.7721   95.2265
                 Rule 6: Off-Peak Hours     0.0009  2.8404    0.0018  5.9173   94.0810
        Rule: Weekend Night Transaction        NaN  0.0000    0.0000  0.0000   99.9981
         Rule: Unusual Hour Transaction     0.0008  1.7042    0.0016  3.9155   96.0827
    Rule: New Account High Amount >$500     0.0063  3.7304    0.0126  1.1111   98.8871
 Rule: Very New Account <7 days Any Txn     0.0018  1.2498    0.0035  1.3336   98.6646
Rule: First Txn <1hr 

## 8. Rank Rules by Performance

In [9]:
# Top rules by Precision
print("\n🏆 TOP RULES BY PRECISION")
print("="*80)
top_precision = rules_df.nlargest(5, 'precision')[['rule_name', 'precision', 'recall', 'f1_score', 'total_flagged']]
print(top_precision.to_string(index=False))

# Top rules by Recall
print("\n\n🏆 TOP RULES BY RECALL")
print("="*80)
top_recall = rules_df.nlargest(5, 'recall')[['rule_name', 'recall', 'precision', 'f1_score', 'total_flagged']]
print(top_recall.to_string(index=False))

# Top rules by F1 Score
print("\n\n🏆 TOP RULES BY F1 SCORE")
print("="*80)
top_f1 = rules_df.nlargest(5, 'f1_score')[['rule_name', 'f1_score', 'precision', 'recall', 'total_flagged']]
print(top_f1.to_string(index=False))

# Lowest FPR
print("\n\n🎯 LOWEST FALSE POSITIVE RATE")
print("="*80)
low_fpr = rules_df.nsmallest(5, 'fpr')[['rule_name', 'fpr', 'precision', 'recall', 'total_flagged']]
print(low_fpr.to_string(index=False))


🏆 TOP RULES BY PRECISION
                      rule_name  precision  recall  f1_score  total_flagged
          Rule 3: High Velocity     0.0179 12.1757    0.0357        3593481
      Rule: PGW NewAcct HighAmt     0.0167  2.2155    0.0331         702434
Rule 1: New Account High Amount     0.0165  1.1930    0.0326         381618
      Rule 5: Multiple Channels     0.0074 18.6707    0.0147       13390487
   Rule: Mobile NewAcct HighAmt     0.0071  1.1551    0.0141         858403


🏆 TOP RULES BY RECALL
                   rule_name  recall  precision  f1_score  total_flagged
Rule: Recipient >$10k In 1hr 86.3473     0.0020    0.0041      222893869
   Rule: High Value txn >$5k 54.4594     0.0069    0.0138       41744625
      Rule: Round Number txn 50.9184     0.0055    0.0110       48903927
 Rule: New Recipient HighAmt 44.2719     0.0049    0.0098       47919217
  Rule: Amount Spike >3x AVG 20.3560     0.0044    0.0089       24201179


🏆 TOP RULES BY F1 SCORE
                      rule_nam

## 9. Summary Statistics

In [10]:
print("\n📊 SUMMARY STATISTICS")
print("="*80)

summary = {
    'Metric': ['Precision (%)', 'Recall (%)', 'F1 Score', 'FPR (%)', 'Accuracy (%)'],
    'Mean': [
        rules_df['precision'].mean(),
        rules_df['recall'].mean(),
        rules_df['f1_score'].mean(),
        rules_df['fpr'].mean(),
        rules_df['accuracy'].mean()
    ],
    'Median': [
        rules_df['precision'].median(),
        rules_df['recall'].median(),
        rules_df['f1_score'].median(),
        rules_df['fpr'].median(),
        rules_df['accuracy'].median()
    ],
    'Std Dev': [
        rules_df['precision'].std(),
        rules_df['recall'].std(),
        rules_df['f1_score'].std(),
        rules_df['fpr'].std(),
        rules_df['accuracy'].std()
    ],
    'Min': [
        rules_df['precision'].min(),
        rules_df['recall'].min(),
        rules_df['f1_score'].min(),
        rules_df['fpr'].min(),
        rules_df['accuracy'].min()
    ],
    'Max': [
        rules_df['precision'].max(),
        rules_df['recall'].max(),
        rules_df['f1_score'].max(),
        rules_df['fpr'].max(),
        rules_df['accuracy'].max()
    ]
}

summary_df = pd.DataFrame(summary)
summary_df = summary_df.round(4)
print(summary_df.to_string(index=False))
print("="*80)


📊 SUMMARY STATISTICS
       Metric    Mean  Median  Std Dev     Min     Max
Precision (%)  0.0056  0.0040   0.0052  0.0008  0.0179
   Recall (%) 15.1418  3.2854  23.1121  0.0000 86.3473
     F1 Score  0.0105  0.0069   0.0104  0.0000  0.0357
      FPR (%)  8.3468  3.4750  16.8612  0.0000 79.4389
 Accuracy (%) 91.6517 96.5233  16.8605 20.5623 99.9981


## 10. Save Results to CSV

In [11]:
# Save to CSV
output_file = '/root/research-dir/dev/jazzcash-fraud-detection/data/fraud_rules_clickhouse_results.csv'
rules_df.to_csv(output_file, index=False)

print(f"\n💾 Results saved to: {output_file}")
print(f"\n📊 DataFrame shape: {rules_df.shape}")
print(f"   • Rows: {rules_df.shape[0]}")
print(f"   • Columns: {rules_df.shape[1]}")


💾 Results saved to: /root/research-dir/dev/jazzcash-fraud-detection/data/fraud_rules_clickhouse_results.csv

📊 DataFrame shape: (22, 13)
   • Rows: 22
   • Columns: 13


## 11. Categorize Rules by Performance

In [12]:
def categorize_rule(row):
    """Categorize rule based on precision and FPR"""
    if row['precision'] >= 80 and row['fpr'] < 5:
        return 'EXCELLENT'
    elif row['precision'] >= 60 and row['fpr'] < 10:
        return 'GOOD'
    elif row['precision'] >= 40 and row['fpr'] < 20:
        return 'MODERATE'
    else:
        return 'POOR'

rules_df['category'] = rules_df.apply(categorize_rule, axis=1)

print("\n🏷️  RULE CATEGORIZATION")
print("="*80)
print(rules_df[['rule_name', 'precision', 'fpr', 'f1_score', 'category']].to_string(index=False))
print("\n")

# Count by category
category_counts = rules_df['category'].value_counts()
print("Category Distribution:")
for category, count in category_counts.items():
    print(f"   • {category}: {count} rules")

print("="*80)


🏷️  RULE CATEGORIZATION
                              rule_name  precision     fpr  f1_score category
        Rule 1: New Account High Amount     0.0165  0.1360    0.0326     POOR
               Rule 2: Very New Account     0.0040  0.0090    0.0066     POOR
                  Rule 3: High Velocity     0.0179  1.2805    0.0357     POOR
                  Rule 4: 3x Max Amount     0.0015 11.6486    0.0030     POOR
              Rule 5: Multiple Channels     0.0074  4.7721    0.0147     POOR
                 Rule 6: Off-Peak Hours     0.0009  5.9173    0.0018     POOR
        Rule: Weekend Night Transaction        NaN  0.0000    0.0000     POOR
         Rule: Unusual Hour Transaction     0.0008  3.9155    0.0016     POOR
    Rule: New Account High Amount >$500     0.0063  1.1111    0.0126     POOR
 Rule: Very New Account <7 days Any Txn     0.0018  1.3336    0.0035     POOR
Rule: First Txn <1hr after Registration     0.0020  0.1768    0.0040     POOR
           Rule: Dormant >30d, High txn

## 12. Close ClickHouse Connection

In [ ]:
# Disconnect from ClickHouse
client.disconnect()

print("✅ ClickHouse connection closed")
print("\n🎉 Analysis Complete!")